<a href="https://colab.research.google.com/github/SuleimanAlbalkhi/explainable-pistol-detection/blob/main/UNet_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/yolo_Dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: yolo_Dataset/images/val/armas (1334)_jpg.rf.f33528c986de7c47ee928d378964b495.jpg  
  inflating: yolo_Dataset/images/val/armas (1336)_jpg.rf.181f5a98eb55b91a13be22f92e1bf529.jpg  
  inflating: yolo_Dataset/images/val/armas (1342)_jpg.rf.3562493a1147f0e9bd800660d3011b2b.jpg  
  inflating: yolo_Dataset/images/val/armas (1345)_jpg.rf.deb1d0de3e2299d09f5e28af432e88eb.jpg  
  inflating: yolo_Dataset/images/val/armas (1351)_jpg.rf.0ac538b0ac3e74393fa7e86085faae6f.jpg  
  inflating: yolo_Dataset/images/val/armas (1353)_jpg.rf.2dcdda58c7901496510de5ab15a2b3b3.jpg  
  inflating: yolo_Dataset/images/val/armas (1357)_jpg.rf.11ecc221735e8c91468106026b345f1d.jpg  
  inflating: yolo_Dataset/images/val/armas (136)_jpg.rf.258e7058c9269f94961eba4ab6ac6654.jpg  
  inflating: yolo_Dataset/images/val/armas (1362)_jpg.rf.dd140589690ff0175ecc9b8e113a3c2e.jpg  
  inflating: yolo_Dataset/images/val/armas (1366)_jpg.rf.57196a4e384c1cc4410e56ebc4fe8

In [ ]:
import os
import random
import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import label

import torchvision.transforms.functional as TF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMG_DIR = "/content/yolo_Dataset/images"
LBL_DIR = "/content/yolo_Dataset/labels"
PRETRAINED = "/content/pistol_detection_cnn.pth"

with open("/content/eval_images.txt") as f:
    selected_images = [line.strip() for line in f]

print(f"🎯 Loaded {len(selected_images)} evaluation images")


Device: cuda
🎯 Loaded 50 evaluation images


In [ ]:
class YoloToMaskDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=416, mask_size=416,
                 blur_kernel=11, blur_sigma=5.0, gamma=5.0,
                 shuffle=False, seed=None):
        """
        img_size: Größe der Eingabebilder (z.B. 416)
        mask_size: Größe der Masken (z.B. 208 = img_size // 2)
        Wir erzeugen "weiche" Gaussian-ähnliche Heatmaps aus den YOLO-Bounding-Boxen.
        """
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.img_size = img_size
        self.mask_size = mask_size
        self.blur_kernel = blur_kernel
        self.blur_sigma = blur_sigma
        self.gamma = gamma

        self.image_files = [
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        self.image_files.sort()

    def __len__(self):
        return len(self.image_files)

    def _load_yolo_labels(self, label_path):
        """YOLO Labels → (class, x_center, y_center, w, h) normalisiert [0,1]."""
        if not os.path.exists(label_path):
            return []

        boxes = []
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, w, h = map(float, parts)
                boxes.append((int(cls), xc, yc, w, h))
        return boxes

    def _yolo_to_xyxy(self, boxes, W, H):
        """Normierte YOLO-Koordinaten → Pixel XYXY im Originalbild."""
        result = []
        for cls, xc, yc, w, h in boxes:
            xc *= W
            yc *= H
            w  *= W
            h  *= H
            xmin = max(0, xc - w/2)
            xmax = min(W-1, xc + w/2)
            ymin = max(0, yc - h/2)
            ymax = min(H-1, yc + h/2)
            result.append((cls, xmin, ymin, xmax, ymax))
        return result

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(
            self.label_dir, os.path.splitext(img_name)[0] + ".txt"
        )

        # Bild laden & auf img_size bringen
        img = Image.open(img_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size))
        img_tensor = TF.to_tensor(img)

        # YOLO-Boxen holen und in Pixel coords
        boxes = self._load_yolo_labels(label_path)

        boxes = self._yolo_to_xyxy(
            boxes,
            self.img_size,
            self.img_size)

        # Heatmap-Maske in mask_size x mask_size
        mask = np.zeros((self.mask_size, self.mask_size), dtype=np.float32)

        # Da mask_size == img_size, ist Skalierung trivial
        sx = self.mask_size / self.img_size
        sy = self.mask_size / self.img_size

        # Rechtecke einzeichnen
        for cls, xmin, ymin, xmax, ymax in boxes:
            xmin_m = int(xmin * sx)
            xmax_m = int(xmax * sx)
            ymin_m = int(ymin * sy)
            ymax_m = int(ymax * sy)
            xmin_m = max(0, xmin_m)
            ymin_m = max(0, ymin_m)
            xmax_m = min(self.mask_size - 1, xmax_m)
            ymax_m = min(self.mask_size - 1, ymax_m)
            if xmax_m > xmin_m and ymax_m > ymin_m:
                mask[ymin_m:ymax_m, xmin_m:xmax_m] = 1.0

        # Gaussian Blur + leichte "Schärfung" über Gamma
        if self.blur_kernel is not None and self.blur_kernel > 1:
            mask = cv2.GaussianBlur(mask, (self.blur_kernel, self.blur_kernel),
                                    self.blur_sigma)
        if mask.max() > 0:
            mask = mask / mask.max()
        if self.gamma is not None and self.gamma > 1.0:
            mask = mask ** self.gamma

        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)  # [1, Hm, Wm]

        return img_tensor, mask_tensor


In [ ]:
class PistolDetectionCNN(nn.Module):
    def __init__(self, num_classes=2, lr=0.001, weight_decay=5e-4):
        super().__init__()

        # --- Block 1 ---
        self.ConvLayer1 = nn.Conv2d(3, 16, 3, padding=1)
        self.BN1 = nn.BatchNorm2d(16)
        self.ReLU1 = nn.ReLU(inplace=True)
        self.Pool1 = nn.MaxPool2d(2, 2)

        # --- Block 2 ---
        self.ConvLayer2 = nn.Conv2d(16, 32, 3, padding=1)
        self.BN2 = nn.BatchNorm2d(32)
        self.ReLU2 = nn.ReLU(inplace=True)
        self.Pool2 = nn.MaxPool2d(2, 2)

        # --- Block 3 ---
        self.ConvLayer3 = nn.Conv2d(32, 64, 3, padding=1)
        self.BN3 = nn.BatchNorm2d(64)
        self.ReLU3 = nn.ReLU(inplace=True)
        self.Pool3 = nn.MaxPool2d(2, 2)

        # --- Block 4 ---
        self.ConvLayer4 = nn.Conv2d(64, 128, 3, padding=1)
        self.BN4 = nn.BatchNorm2d(128)
        self.ReLU4 = nn.ReLU(inplace=True)
        self.Pool4 = nn.MaxPool2d(2, 2)

        # --- Block 5 ---
        self.ConvLayer5 = nn.Conv2d(128, 256, 3, padding=1)
        self.BN5 = nn.BatchNorm2d(256)
        self.ReLU5 = nn.ReLU(inplace=True)
        self.Pool5 = nn.MaxPool2d(2, 2)

        # --- Block 6 ---
        self.ConvLayer6 = nn.Conv2d(256, 416, 3, padding=1)
        self.BN6 = nn.BatchNorm2d(416)
        self.ReLU6 = nn.ReLU(inplace=True)
        self.Pool6 = nn.MaxPool2d(2, 2)  # wird im UNet-Encoder NICHT verwendet

        # Klassifikationskopf (für Pretraining – im UNet nicht verwendet)
        self.GlobalPool = nn.AdaptiveAvgPool2d(1)
        self.Flatten = nn.Flatten()
        self.FC1 = nn.Linear(416, 128)
        self.ReLU_fc = nn.ReLU(inplace=True)
        self.Dropout = nn.Dropout(0.1)
        self.FC2 = nn.Linear(128, num_classes)

    def forward_features(self, x):
        # ursprünglicher Feature-Pfad (mit Pool6) – wird vom Klassifikations-Forward benutzt,
        # aber nicht für das UNet
        x = self.Pool1(self.ReLU1(self.BN1(self.ConvLayer1(x))))
        x = self.Pool2(self.ReLU2(self.BN2(self.ConvLayer2(x))))
        x = self.Pool3(self.ReLU3(self.BN3(self.ConvLayer3(x))))
        x = self.Pool4(self.ReLU4(self.BN4(self.ConvLayer4(x))))
        x = self.Pool5(self.ReLU5(self.BN5(self.ConvLayer5(x))))
        x = self.Pool6(self.ReLU6(self.BN6(self.ConvLayer6(x))))
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.GlobalPool(x)
        x = self.Flatten(x)
        x = self.ReLU_fc(self.FC1(x))
        x = self.Dropout(x)
        x = self.FC2(x)
        return x

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_c + skip_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

# -------------------------------------------------------------
# ENCODER FÜR UNET (mit c1..c6, ohne Pool6 – Bottleneck bleibt 13x13)
# -------------------------------------------------------------
class Encoder(nn.Module):
    def __init__(self, cnn):
        super().__init__()
        self.cnn = cnn

    def forward(self, x):
        # Block 1: 416 → 208
        x = self.cnn.Pool1(self.cnn.ReLU1(self.cnn.BN1(self.cnn.ConvLayer1(x))))
        c1 = x  # [B,16,208,208]

        # Block 2: 208 → 104
        x = self.cnn.Pool2(self.cnn.ReLU2(self.cnn.BN2(self.cnn.ConvLayer2(x))))
        c2 = x  # [B,32,104,104]

        # Block 3: 104 → 52
        x = self.cnn.Pool3(self.cnn.ReLU3(self.cnn.BN3(self.cnn.ConvLayer3(x))))
        c3 = x  # [B,64,52,52]

        # Block 4: 52 → 26
        x = self.cnn.Pool4(self.cnn.ReLU4(self.cnn.BN4(self.cnn.ConvLayer4(x))))
        c4 = x  # [B,128,26,26]

        # Block 5: 26 → 13
        x = self.cnn.Pool5(self.cnn.ReLU5(self.cnn.BN5(self.cnn.ConvLayer5(x))))
        c5 = x  # [B,256,13,13]

        # Block 6: Conv6 nur auf 13×13 (KEIN Pool6!)
        x = self.cnn.ReLU6(self.cnn.BN6(self.cnn.ConvLayer6(x)))
        c6 = x  # [B,416,13,13]

        return c1, c2, c3, c4, c5, c6


# -------------------------------------------------------------
# SYMMETRISCHER DECODER MIT c6, OUTPUT 416x416
# -------------------------------------------------------------
class Decoder(nn.Module): # <--- Added missing class definition
    def __init__(self):
        super().__init__()

        # Bottleneck (416 Kanäle, 13×13)
        self.bottleneck = DoubleConv(416, 416)

        # up6: 13 → 26, Skip: c5 (256 Kanäle)
        self.up6 = DecoderBlock(in_c=416, skip_c=256, out_c=256)

        # up5: 26 → 52, Skip: c4 (128 Kanäle)
        self.up5 = DecoderBlock(in_c=256, skip_c=128, out_c=128)

        # up4: 52 → 104, Skip: c3 (64 Kanäle)
        self.up4 = DecoderBlock(in_c=128, skip_c=64, out_c=64)

        # up3: 104 → 208, Skip: c2 (32 Kanäle)
        self.up3 = DecoderBlock(in_c=64,  skip_c=32, out_c=32)

        # up2: 208 → 416, Skip: c1 (16 Kanäle)
        self.up2 = DecoderBlock(in_c=32,  skip_c=16, out_c=16)

        # up1: letzte UpSampling-Stufe → 416×416 Output passend machen
        self.up1 = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)

        # finale 1×1-Conv: Heatmap
        self.final = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, c1, c2, c3, c4, c5, c6):

        # Bottleneck bleibt 13×13
        b = self.bottleneck(c6)

        # Up-Pfad
        x = self.up6(b,  c5)   # 13 → 26
        x = self.up5(x, c4)    # 26 → 52
        x = self.up4(x, c3)    # 52 → 104
        x = self.up3(x, c2)    # 104 → 208
        x = self.up2(x, c1)    # 208 → 416

        # nochmal upsampling, ohne Skip
        x = self.up1(x)        # 416 bleibt 416

        # finale Heatmap
        return self.final(x)

# -------------------------------------------------------------
# UNET-GESAMTMODELL
# -------------------------------------------------------------
class UNet(nn.Module):
    def __init__(self, cnn: PistolDetectionCNN):
        super().__init__()
        self.encoder = Encoder(cnn)
        self.decoder = Decoder()

    def forward(self, x):
        c1, c2, c3, c4, c5, c6 = self.encoder(x)
        return self.decoder(c1, c2, c3, c4, c5, c6)

In [ ]:
def load_encoder_weights(unet_model, pretrained_path):

    if not os.path.exists(pretrained_path):
        print("WARNUNG: Kein Pretrained gefunden.")
        return

    checkpoint = torch.load(pretrained_path, map_location="cpu")
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state = checkpoint["state_dict"]
    else:
        state = checkpoint

    enc_state = unet_model.encoder.state_dict()
    new_weights = {}

    for k_pre, v_pre in state.items():
        target = "cnn." + k_pre
        if target in enc_state and enc_state[target].shape == v_pre.shape:
            new_weights[target] = v_pre

    enc_state.update(new_weights)
    unet_model.encoder.load_state_dict(enc_state, strict=False)
    print(f"{len(new_weights)} Layer erfolgreich geladen.")

In [ ]:
def train_one_epoch(model, loader, opt, crit):
    model.train()
    total = 0
    for img, m in loader:
        img = img.to(device)
        m = m.to(device)
        opt.zero_grad()
        out = model(img)
        loss = crit(out, m)
        loss.backward()
        opt.step()
        total += loss.item() * img.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, crit):
    model.eval()
    total = 0
    for img, m in loader:
        img = img.to(device)
        m = m.to(device)
        out = model(img)
        loss = crit(out, m)
        total += loss.item() * img.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def heatmap_to_boxes(logits, img_size=416, threshold=0.5, min_area=30):
    probs = torch.sigmoid(logits)[0,0].cpu().numpy()
    Hm, Wm = probs.shape
    mask = (probs > threshold).astype(np.uint8)
    labeled, num = label(mask)
    boxes = []

    for i in range(1, num+1):
        coords = np.column_stack(np.where(labeled == i))
        if coords.shape[0] < min_area:
            continue

        ys, xs = coords[:,0], coords[:,1]
        xmin_m, xmax_m = xs.min(), xs.max()
        ymin_m, ymax_m = ys.min(), ys.max()

        # Confidence = max probability in der Region
        conf = probs[labeled == i].max()

        sx = img_size / Wm
        sy = img_size / Hm
        xmin = int(xmin_m * sx)
        xmax = int((xmax_m+1) * sx)
        ymin = int(ymin_m * sy)
        ymax = int((ymax_m+1) * sy)

        boxes.append((xmin, ymin, xmax, ymax, conf))  # ✅ 5 Werte

    return boxes, probs


def visualize_reconstruction(
    img_tensor,
    mask_gt,
    probs,
    boxes,
    img_size=416,
    save_path=None
):
    """
    Reihenfolge:
    1. Original + GT Box
    2. Original + Pred Box
    3. Pred Heatmap Overlay
    """

    img_np = img_tensor.permute(1,2,0).cpu().numpy()

    probs_resized = cv2.resize(probs, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    if mask_gt is not None:
        gt_mask = mask_gt[0].cpu().numpy()
        gt_resized = cv2.resize(gt_mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
    else:
        gt_resized = None

    # 🔹 Figure explizit erstellen
    fig = plt.figure(figsize=(18,6))

    # ---------------------------------------------------------
    # 1. Original + GT Box
    # ---------------------------------------------------------
    ax1 = fig.add_subplot(1,3,1)
    ax1.imshow(img_np)

    if gt_resized is not None:
        ys, xs = np.where(gt_resized > 0.1)
        if len(xs) > 0 and len(ys) > 0:
            xmin, xmax = xs.min(), xs.max()
            ymin, ymax = ys.min(), ys.max()
            rect = plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                 fill=False, edgecolor="red", linewidth=2)
            ax1.add_patch(rect)

    ax1.set_title("Original + GT Box")
    ax1.axis("off")

    # ---------------------------------------------------------
    # 2. Original + PRED Box
    # ---------------------------------------------------------
    ax2 = fig.add_subplot(1,3,2)
    ax2.imshow(img_np)

    for box in boxes:
        if len(box) == 5:
            xmin, ymin, xmax, ymax, conf = box
        else:
            xmin, ymin, xmax, ymax = box

        rect = plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                             fill=False, edgecolor="cyan", linewidth=2)
        ax2.add_patch(rect)

    ax2.set_title("Original + Predicted Box")
    ax2.axis("off")

    # ---------------------------------------------------------
    # 3. Heatmap Overlay
    # ---------------------------------------------------------
    ax3 = fig.add_subplot(1,3,3)
    ax3.imshow(img_np, alpha=0.6)
    ax3.imshow(probs_resized, cmap="jet", alpha=0.4)
    ax3.set_title("Predicted Heatmap")
    ax3.axis("off")

    fig.tight_layout()

    # 🔹 Speichern, wenn Pfad übergeben wurde
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.close(fig)  # wichtig für Speicher

    return save_path

In [ ]:
import os

# Create paths for train and validation datasets
IMG_DIR_TRAIN = os.path.join(IMG_DIR, "train")
LBL_DIR_TRAIN = os.path.join(LBL_DIR, "train")


IMG_DIR_VAL = os.path.join(IMG_DIR, "val")
LBL_DIR_VAL = os.path.join(LBL_DIR, "val")

IMG_DIR_TEST = os.path.join(IMG_DIR, "test")
LBL_DIR_TEST = os.path.join(LBL_DIR, "test")

# Instantiate datasets using pre-defined splits
train_ds = YoloToMaskDataset(IMG_DIR_TRAIN, LBL_DIR_TRAIN, img_size=416, mask_size=416, shuffle=True, seed=42)
val_ds   = YoloToMaskDataset(IMG_DIR_VAL, LBL_DIR_VAL, img_size=416, mask_size=416,  shuffle=True, seed=42)
test_ds  = YoloToMaskDataset(IMG_DIR_TEST, LBL_DIR_TEST, img_size=416, mask_size=416,  shuffle=True, seed=42)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=2)

In [ ]:

# -------------------------------------------------------------
# MODELL, LOSS, OPTIMIERER
# -------------------------------------------------------------
cnn = PistolDetectionCNN()
model = UNet(cnn).to(device)

load_encoder_weights(model, PRETRAINED)

pos_weight = torch.tensor([5.0], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=5e-5)

# -------------------------------------------------------------
# TRAINING LOOP MIT EARLY STOPPING
# -------------------------------------------------------------
best_val = float("inf")
best_state = None
patience = 5
trigger = 0
EPOCHS = 40

for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate(model, val_loader, criterion)

    print(f"Epoch {epoch:02d} | Train {tr_loss:.4f} | Val {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        trigger = 0
        print("  → New best model")
    else:
        trigger += 1
        if trigger >= patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
model.to(device)

torch.save(model.state_dict(), "/content/unet_pistol_detection.pth")
print("Best model saved to /content/unet_pistol_detection")


46 Layer erfolgreich geladen.


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "/content/unet_pistol_detection-final.pth")

# Loading a trained Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = PistolDetectionCNN()
model = UNet(cnn)

model.load_state_dict(torch.load("/content/unet_pistol_detection-final.pth", map_location=device))

model.to(device)
model.eval()

UNet(
  (encoder): Encoder(
    (cnn): PistolDetectionCNN(
      (ConvLayer1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (BN1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (ReLU1): ReLU(inplace=True)
      (Pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (ConvLayer2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (BN2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (ReLU2): ReLU(inplace=True)
      (Pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (ConvLayer3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (BN3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (ReLU3): ReLU(inplace=True)
      (Pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (ConvLayer4): Conv2d(64, 12

In [ ]:
@torch.no_grad()
def evaluate_examples(model, dataset, selected_images, output_dir="unet_outputs"):
    model.eval()

    os.makedirs(output_dir, exist_ok=True)

    # Map filename → index (nur Basename verwenden!)
    name_to_idx = {
        os.path.basename(dataset.image_files[i]): i
        for i in range(len(dataset))
    }

    for i, full_path in enumerate(selected_images):
        filename = os.path.basename(full_path)

        if filename not in name_to_idx:
            print(f"{filename} not found in dataset")
            continue

        idx = name_to_idx[filename]

        img, mask = dataset[idx]
        logits = model(img.unsqueeze(0).to(device))
        boxes, probs = heatmap_to_boxes(logits)

        print(f"Sample {i+1}: {len(boxes)} boxes")

        save_path = os.path.join(
            output_dir,
            f"{i+1:03d}_{filename}"
        )

        visualize_reconstruction(
            img,
            mask,
            probs,
            boxes,
            save_path=save_path
        )

evaluate_examples(model, test_ds, selected_images)


Sample 1: 1 boxes
Sample 2: 1 boxes
Sample 3: 1 boxes
Sample 4: 1 boxes
Sample 5: 1 boxes
Sample 6: 1 boxes
Sample 7: 1 boxes
Sample 8: 1 boxes
Sample 9: 2 boxes
Sample 10: 0 boxes
Sample 11: 1 boxes
Sample 12: 10 boxes
Sample 13: 0 boxes
Sample 14: 1 boxes
Sample 15: 1 boxes
Sample 16: 3 boxes
Sample 17: 1 boxes
Sample 18: 11 boxes
Sample 19: 1 boxes
Sample 20: 0 boxes
Sample 21: 2 boxes
Sample 22: 0 boxes
Sample 23: 1 boxes
Sample 24: 0 boxes
Sample 25: 1 boxes
Sample 26: 1 boxes
Sample 27: 0 boxes
Sample 28: 0 boxes
Sample 29: 1 boxes
Sample 30: 1 boxes
Sample 31: 1 boxes
Sample 32: 0 boxes
Sample 33: 2 boxes
Sample 34: 8 boxes
Sample 35: 1 boxes
Sample 36: 0 boxes
Sample 37: 6 boxes
Sample 38: 0 boxes
Sample 39: 1 boxes
Sample 40: 1 boxes
Sample 41: 1 boxes
Sample 42: 0 boxes
Sample 43: 1 boxes
Sample 44: 12 boxes
Sample 45: 3 boxes
Sample 46: 0 boxes
Sample 47: 0 boxes
Sample 48: 0 boxes
Sample 49: 0 boxes
Sample 50: 2 boxes


In [ ]:

!zip -r /content/unet_outputs.zip /content/unet_outputs


  adding: content/unet_outputs/ (stored 0%)
  adding: content/unet_outputs/033_armas (1429)_jpg.rf.83bfc05ab9fad72f189690c5f7666241.jpg (deflated 5%)
  adding: content/unet_outputs/038_21eb99457fc91970.jpg (deflated 3%)
  adding: content/unet_outputs/042_16010699e82ef96e.jpg (deflated 6%)
  adding: content/unet_outputs/044_armas (2822)_jpg.rf.8199a04cf8290f4269713b3bffa3fd67.jpg (deflated 5%)
  adding: content/unet_outputs/011_armas (1157)_jpg.rf.51f7d3768ca3c6e987960cf0cda5544d.jpg (deflated 17%)
  adding: content/unet_outputs/002_armas (1148)_jpg.rf.b989e1fd209bb6c202bcae58d41fa93f.jpg (deflated 7%)
  adding: content/unet_outputs/008_armas (100)_jpg.rf.6b05c1801376fcfe8912599edf7ac2b1.jpg (deflated 17%)
  adding: content/unet_outputs/028_23eab470ddaec483.jpg (deflated 5%)
  adding: content/unet_outputs/016_armas (97)_jpg.rf.7c20b841ec40e6ad328848b0ce82fab7.jpg (deflated 9%)
  adding: content/unet_outputs/020_9248e5fa-video2_06b23f50.jpg (deflated 5%)
  adding: content/unet_outputs/04

# ***Evaluation***

In [ ]:
def box_iou(box1, box2):
    """
    box = (xmin, ymin, xmax, ymax) or (xmin, ymin, xmax, ymax, conf)
    """
    # FIXED: Handle boxes with confidence
    box1 = box1[:4] if len(box1) > 4 else box1
    box2 = box2[:4] if len(box2) > 4 else box2

    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter_area + 1e-6
    return inter_area / union

In [ ]:
def load_gt_boxes(label_path, img_size=416):
    boxes = []

    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            cls, xc, yc, w, h = map(float, line.split())

            xc *= img_size
            yc *= img_size
            w  *= img_size
            h  *= img_size

            xmin = int(xc - w / 2)
            ymin = int(yc - h / 2)
            xmax = int(xc + w / 2)
            ymax = int(yc + h / 2)

            boxes.append((xmin, ymin, xmax, ymax))

    return boxes



In [ ]:
def average_precision(pred_boxes, gt_boxes, iou_thresh=0.5):
    """
    pred_boxes: [(xmin,ymin,xmax,ymax,conf), ...]
    gt_boxes: [(xmin,ymin,xmax,ymax), ...]
    """

    if len(gt_boxes) == 0:
        return 1.0 if len(pred_boxes) == 0 else 0.0

    # Sort predictions by confidence
    pred_boxes = sorted(pred_boxes, key=lambda x: x[4], reverse=True)

    tp = np.zeros(len(pred_boxes))
    fp = np.zeros(len(pred_boxes))
    matched_gt = set()

    for i, p in enumerate(pred_boxes):
        best_iou = 0.0
        best_gt = -1

        for j, g in enumerate(gt_boxes):
            iou = box_iou(p[:4], g)
            if iou > best_iou:
                best_iou = iou
                best_gt = j

        if best_iou >= iou_thresh and best_gt not in matched_gt:
            tp[i] = 1
            matched_gt.add(best_gt)
        else:
            fp[i] = 1

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)

    recall = tp_cum / (len(gt_boxes) + 1e-6)
    precision = tp_cum / (tp_cum + fp_cum + 1e-6)

    # AP = area under PR curve
    return np.trapezoid(precision, recall)



In [ ]:
def match_boxes(pred_boxes, gt_boxes, iou_thresh=0.5):
    matched_gt = set()
    tp = fp = 0

    pred_boxes = sorted(pred_boxes, key=lambda x: x[4], reverse=True)

    for p in pred_boxes:
        best_iou = 0
        best_gt = -1

        for i, g in enumerate(gt_boxes):
            iou = box_iou(p[:4], g)
            if iou > best_iou:
                best_iou = iou
                best_gt = i

        if best_iou >= iou_thresh and best_gt not in matched_gt:
            tp += 1
            matched_gt.add(best_gt)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn

In [ ]:
@torch.no_grad()
def evaluate_detection_metrics(
    model,
    dataset,
    lbl_dir,
    img_size=416,
    heatmap_thresh=0.5,
    iou_thresholds=np.arange(0.5, 0.96, 0.05)
):
    """
    Dataset-level evaluation.
    Computes true mAP@50 and mAP@50–95 (COCO-style),
    plus Precision, Recall, F1 and Mean IoU.
    Now also includes Image-level True Negatives (TN_images).
    """

    model.eval()
    device = next(model.parameters()).device

    # --------------------------------------------------
    # Global collectors
    # --------------------------------------------------
    all_predictions = {t: [] for t in iou_thresholds}  # t → [(conf, TP/FP)]
    total_gt_boxes = 0
    ious = []

    # New: Image-level counters for True Negatives
    tn_images = 0
    total_images = 0

    # --------------------------------------------------
    # Iterate dataset
    # --------------------------------------------------
    indices = list(range(len(dataset)))
    random.shuffle(indices)

    for idx in indices:
        total_images += 1
        img, _ = dataset[idx]

        img_name = dataset.image_files[idx]
        label_path = os.path.join(
            lbl_dir, img_name.rsplit(".", 1)[0] + ".txt"
        )

        gt_boxes = load_gt_boxes(label_path, img_size)
        total_gt_boxes += len(gt_boxes)

        logits = model(img.unsqueeze(0).to(device))
        pred_boxes, _ = heatmap_to_boxes(
            logits,
            img_size=img_size,
            threshold=heatmap_thresh
        )

        # Image-level True Negatives: No GT boxes and no predicted boxes
        if len(gt_boxes) == 0 and len(pred_boxes) == 0:
            tn_images += 1

        # Sort predictions by confidence (descending)
        pred_boxes = sorted(pred_boxes, key=lambda x: x[4], reverse=True)
        # --------------------------------------------------
        # TP-based IoU collection (IoU threshold = 0.5)
        # --------------------------------------------------
        matched_gt_iou = set()

        for p in pred_boxes:
            best_iou = 0.0
            best_gt = -1

            for j, g in enumerate(gt_boxes):
                iou = box_iou(p[:4], g)
                if iou > best_iou:
                    best_iou = iou
                    best_gt = j

            if best_iou >= 0.5 and best_gt not in matched_gt_iou:
                ious.append(best_iou)
                matched_gt_iou.add(best_gt)


        # --------------------------------------------------
        # Matching for each IoU threshold
        # --------------------------------------------------
        for t in iou_thresholds:
            matched_gt = set()

            for p in pred_boxes:
                best_iou = 0.0
                best_gt = -1

                for j, g in enumerate(gt_boxes):
                    iou = box_iou(p[:4], g)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt = j

                if best_iou >= t and best_gt not in matched_gt:
                    all_predictions[t].append((p[4], 1))  # TP
                    matched_gt.add(best_gt)
                else:
                    all_predictions[t].append((p[4], 0))  # FP

    # --------------------------------------------------
    # AP computation per IoU threshold
    # --------------------------------------------------
    APs = {}

    for t in iou_thresholds:
        preds = all_predictions[t]

        if len(preds) == 0 and total_gt_boxes == 0: # If no predictions and no ground truth, AP is 1
            APs[t] = 1.0
            continue
        elif len(preds) == 0 or total_gt_boxes == 0: # If only predictions or only ground truth, AP is 0
            APs[t] = 0.0
            continue

        # Global sort by confidence
        preds.sort(key=lambda x: x[0], reverse=True)

        tp = np.array([p[1] for p in preds])
        fp = 1 - tp

        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)

        recall = tp_cum / (total_gt_boxes + 1e-6)
        precision = tp_cum / (tp_cum + fp_cum + 1e-6)

        APs[t] = float(np.trapz(precision, recall)) if len(recall) > 1 else 0.0

    # --------------------------------------------------
    # Final metrics
    # --------------------------------------------------
    mAP50 = APs[0.5]
    mAP5095 = float(np.mean(list(APs.values()))) if APs else 0.0

    # Precision / Recall / F1 @ IoU=0.5 (object-level)
    preds_50 = all_predictions[0.5]
    preds_50.sort(key=lambda x: x[0], reverse=True)

    tp = np.array([p[1] for p in preds_50])
    fp = 1 - tp

    TP = int(tp.sum())
    FP = int(fp.sum())
    FN = int(total_gt_boxes - TP)

    # Re-calculate Precision, Recall, and F1-score specifically for IoU=0.5
    final_precision = TP / (TP + FP + 1e-6) if (TP + FP) > 0 else 0.0
    final_recall = TP / (TP + FN + 1e-6) if (TP + FN) > 0 else 0.0
    final_f1_score = 2 * final_precision * final_recall / (final_precision + final_recall + 1e-6) if (final_precision + final_recall) > 0 else 0.0

    return {
        "mAP@50":    mAP50,
        "mAP@50-95": mAP5095,
        "Mean IoU":  float(np.mean(ious)) if ious else 0.0,
        "Precision": final_precision,
        "Recall":    final_recall,
        "F1-score":  final_f1_score,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN ": tn_images
    }

In [ ]:
metrics = evaluate_detection_metrics(
    model,
    test_ds,
    LBL_DIR_TEST,
    img_size=416,
    heatmap_thresh=0.5
)

for k, v in metrics.items():
    print(f"{k:19s}: {v:.4f}" if isinstance(v, float) else f"{k:19s}: {v}")

mAP@50             : 0.4058
mAP@50-95          : 0.2198
Mean IoU           : 0.7963
Precision          : 0.2562
Recall             : 0.5000
F1-score           : 0.3388
TP                 : 166
FP                 : 482
FN                 : 166
TN                 : 124


/tmp/ipykernel_1044/3833453075.py:131: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  APs[t] = float(np.trapz(precision, recall)) if len(recall) > 1 else 0.0
